In [1]:
import torch
import numpy as np
import pandas as pd

from helpers.persistence import *
from helpers.openml_data_v2 import openml_cc18_list, get_data1
from helpers.openml_data import tabular_id_list
from helpers.progress_bar import ProgressBar
from itertools import product
import helpers.trainer_multiworker as trainer

In [2]:
df, labels, cats = get_data1(1510)

In [10]:
import random

def set_random_seeds():
    torch.manual_seed(0)
    np.random.seed(0)
    random.seed(0) # mcar sampler uses random maybe   
    
set_random_seeds()

In [11]:
if torch.cuda.is_available():
    device = "cuda:0"
else:
    device = "cpu"

# device = 'cpu'
trainer.device = device
print(device)

cuda:0


In [12]:
expt = 'test-time'

save_path, export_path = f'./saved_vars/{expt}.pkl', f'./exports/{expt}.csv'

dataset_results = load_var(save_path) or {}
dataset_results = {}
# len(dataset_results)

Could not load ./saved_vars/test-time.pkl because [Errno 2] No such file or directory: './saved_vars/test-time.pkl'


In [13]:
# dataset_results

In [14]:
# # rename
# r = {}
# r['1510_mixup-draw-features_loss=simclr_patience=9999'] = dataset_results['1510_mixup-draw_loss=simclr_patience=9999']
# save_var(r, save_path)

In [15]:
# from tqdm import tqdm
# pbar = tqdm(list(range(10)))

# for i in pbar:
#     pbar.set_description(''.join(['+' if x%10!=0 else '|' for x in range(1,251)]))    
    
# # print(''.join('+' if x%10!=0 else '|' for x in range(1,251)))

In [16]:

names = openml_cc18_list[:1]
names = tabular_id_list[10:15]
corruptions = [
    'draw',
    'draw-feature',
    'pass',
    'noise',
    'sample'
]

names = [1510]

n_pairs = [128]
missing_rate = 0.6
missing_type = 'mcar'
patience = 9999

losses = ['simclr','scarf','bad-scarf']
losses = ['simclr']

expt_list = list(product(names, corruptions, losses))
pbar = ProgressBar(expt_list)
trainer.pbar = pbar

for dataset_name, c, loss in pbar:     
    pbar.clear_prefix()
    
    trainer.settings['corruptor2'].update({
        'method': c,
        'corruption_rate': 0.6,
        'missing': missing_rate,
        'missing_type': missing_type,
        'mice': 'LinearRegression',
    })
    trainer.settings.update({
        'method': 'bootstrap', 
        'pretrain_loss':loss,
        'corrupt_before_ohe': c in ['draw','draw-feature'],
        'test_pretrain': False,
        'patience': patience,
        'train_workers': 0,
        'bootstrap': 10,
    })
    
    # print(trainer.settings)
    
    corruption_key = f"{c}-{missing_type}-{missing_rate}" if c in ['knn', 'mice'] else c
    key = f'{dataset_name}_{corruption_key}_loss={loss}_patience={patience}'
    expt1 = f'{expt}_{key}'
    
    pbar.add_prefix(key)

    if key in dataset_results.keys():
        # print(key, 'done')
        continue
        
    pbar.set_description('')

    # print(key, 'doing')
    set_random_seeds()
    scores, z_scores, p_scores, pretrain_time = trainer.do_all(dataset_name, expt=expt1)

    dataset_results[key] = {
        'scores': scores,
        'z_scores': z_scores,
        'p_scores': p_scores,
        'pretrain_time': pretrain_time,
    }
    save_var(dataset_results, save_path)


1510_draw_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ; mean_va

test accuracy: 0.9675  ( 0.0153 )
time taken, total=2346.71s or 234.67s per fold


1510_draw-feature_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ;

test accuracy: 0.9596  ( 0.0204 )
time taken, total=2372.98s or 237.30s per fold


1510_pass_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ; mean_va

...

1510_noise_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ; mean_v

test accuracy: 0.9552  ( 0.0206 )
time taken, total=2353.16s or 235.32s per fold


1510_sample_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ; mean_

test accuracy: 0.957  ( 0.016 )
time taken, total=2388.60s or 238.86s per fold


1510_sample_loss=simclr_patience=9999 | fold=10/10 |  Finetuning | epoch: 200/200;mean_loss: 0.000 ; mean_


In [17]:
cols = [ 'dataset', 'corruption', 'npairs', 'patience',
        'fold', 'test_score', 
        'z_score_LR', 'p_score_LR', 
        'z_score_GBT', 'p_score_GBT', 
        'avg_loss_time']
rows = []

for k,v in dataset_results.items():
    common_data = k.split('_')
    
    # common_data[-1] = np.round(float(common_data[-1]), 5)
    n_folds = len(v['scores'])
    
    for i in range(n_folds):
        avg_time = np.mean(v['pretrain_time'][i])
        base_scores = [
            v['z_scores']['Logistic Regression'][i], 
            v['p_scores']['Logistic Regression'][i], 
            v['z_scores']['Gradient Boosting'][i], 
            v['p_scores']['Gradient Boosting'][i]
        ] if len(v['z_scores']['Logistic Regression'])>0 else [0,0,0,0]
        row = common_data[:-1] + [int(common_data[-1].split('=')[1])] + [i, v['scores'][i], 
                             *base_scores, 
                             avg_time]
        rows.append(row)
        # print(row)
        # break
        
df = pd.DataFrame(rows, columns=cols)
df.to_csv(export_path)

In [18]:
tmp = df.groupby(['dataset', 'corruption'])[[
    'test_score', 
    'z_score_LR', 'p_score_LR', 
    'z_score_GBT', 'p_score_GBT', 
    'avg_loss_time']].agg('mean')
tmp

test_score  z_score_LR  p_score_LR  z_score_GBT  \
dataset corruption                                                      
1510    draw            0.967481         0.0         0.0          0.0   
        draw-feature    0.959640         0.0         0.0          0.0   
        noise           0.955245         0.0         0.0          0.0   
        pass            0.966612         0.0         0.0          0.0   
        sample          0.956986         0.0         0.0          0.0   

                      p_score_GBT  avg_loss_time  
dataset corruption                                
1510    draw                  0.0       0.013108  
        draw-feature          0.0       0.013033  
        noise                 0.0       0.013179  
        pass                  0.0       0.013144  
        sample                0.0       0.013148